<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/02_criar_view.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Criação da View de Análise no BigQuery
Este notebook consolida as tabelas importadas em uma única visão (View) com nomes de colunas formatados em português, pronta para o Looker Studio / Power BI.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

# Autenticação no Google Colab
auth.authenticate_user()

In [ ]:
project_id = 'directed-mender-489100-m3'
dataset_id = 'games_data'
client = bigquery.Client(project=project_id)

In [ ]:
view_id = f"{project_id}.{dataset_id}.vw_analise_games"

sql_view = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    g.id AS game_id,
    g.name AS nome_jogo,
    g.released AS data_lancamento,
    g.rating AS nota_usuarios,
    g.metacritic AS nota_critica,
    p.publisher_name AS distribuidora,
    gen.genre_name AS genero,
    plat.platform_name AS plataforma,
    m.completability_index AS indice_conclusao,
    m.consensus_score AS score_consenso
FROM
    `{project_id}.{dataset_id}.games` g
LEFT JOIN
    `{project_id}.{dataset_id}.game_publishers` p ON CAST(g.id AS STRING) = CAST(p.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_id}.game_genres` gen ON CAST(g.id AS STRING) = CAST(gen.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_id}.game_platforms` plat ON CAST(g.id AS STRING) = CAST(plat.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_id}.game_derived_metrics` m ON CAST(g.id AS STRING) = CAST(m.game_id AS STRING)
"""

print(f"Criando a view {view_id}...")
query_job = client.query(sql_view)
query_job.result()
print(f"✅ Sucesso! View criada e pronta para uso no BI.")